<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell

Run the setup cell below to import the libraries and load the pre-built vector store from the setup module. This workspace includes two pre-configured files: 

- `sample_document.txt` contains an overview of LangChain and its components. 
- `langchain_setup.py` loads `sample_document.txt`, splits it into chunks, converts them into embeddings, and stores them in a FAISS vector store. The `vector_store` object is imported from this module and is ready for use. 

The imported libraries are used to build the RAG chain: 

- `ChatOpenAI` is the language model that generates answers from retrieved context. 
- `ChatPromptTemplate` structures the prompt sent to the LLM. 
- `RunnablePassthrough` passes input through the chain without modification. 
- `StrOutputParser` converts the model's response into plain text.

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key.

In [ ]:
# Third-party imports
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Local application imports
from langchain_setup import vector_store

**Note:** Be sure to run the Setup cell above before beginning with the checkpoints!

#### Checkpoint 1/4: Creating a Retriever

Before building the RAG chain, we need a retriever to automatically fetch relevant chunks from the vector store based on the user's query.  

Convert `vector_store` into a retriever named `retriever` and set `k` to `3` to retrieve the top 3 most relevant chunks for each query. 

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Convert the vector store into a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

#### Testing the Retriever

Run the code below to confirm the retriever is working correctly before building the full RAG chain.

In [ ]:
# Test the retriever
test_docs = retriever.invoke("What are the main components of LangChain?")

print(f"Retrieved {len(test_docs)} chunks:\n")
for i, doc in enumerate(test_docs, 1):
    print(f"Chunk {i}: {doc.page_content[:100]}...")

#### Checkpoint 2/4: Creating the Prompt Template and LLM

With the retriever ready, we can now define how the model receives and processes information. 

Create a prompt template named `prompt` that structures how the context and question are presented to the LLM. The template consists of three components: 

- An instruction that describes the assistant as helpful and directs it to use the provided context when answering. 
- A `context` placeholder to insert the retrieved information. 
- A `question` placeholder for the user's query. 

Create a `ChatOpenAI` instance named `llm` with model set to `"gpt-4o-mini"` and `temperature` set to `0` for deterministic responses. 

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Create the prompt template 
prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Use the context below to answer the question.
    Context: {context}
    Question: {question}
    Answer:"""
)

# Initialize the LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

#### Checkpoint 3/4: Building the RAG Chain

With the retriever, prompt, and LLM in place, we can now wire everything together into a complete RAG chain named `rag_chain`. 

- Map `context` to `retriever` to automatically fetch relevant chunks for each query. 
- Map `question` to `RunnablePassthrough()` to pass the user's input through the chain unchanged. 
- Connect the `prompt`, `llm`, and `StrOutputParser` in sequence using the pipe operator to format, generate, and parse the response. 

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Build the RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

#### Checkpoint 4/4: Asking Questions Using Retrieved Context

The RAG chain is now ready. When it receives a question, it automatically retrieves relevant chunks, formats them into a prompt, sends the prompt to the LLM, and returns the answer as plain text. 

- Invoke `rag_chain` with the question `"What are the main components of LangChain?"` and store the result in a variable named `answer` to capture the generated response.  
- Uncomment the `print()` statements to display the question and answer. 

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Invoke the RAG chain
answer = rag_chain.invoke(
    "What are the main components of LangChain?"
)

print(f"Question: What are the main components of LangChain?\n")
print(f"Answer:\n{answer}")

<details> 

<summary><strong>Observations</strong></summary> 

The answer is grounded in the retrieved context from `sample_document.txt`. The LLM did not generate this information from general knowledge — it used the chunks retrieved by the vector store to produce a precise, document-based response. 

In the next exercise, we will combine the agent and the RAG pipeline into one workflow. Instead of retrieving context every time, we will turn the retriever into a tool so the agent can decide when it is needed. This lets us compare selective retrieval with always-on RAG. 

</details> 